# Stage 0 — Stratified sampling

**Goal.** Download a stratified sample of **no-photo** CV images from PopResume (800 per occupation, seed 42), save the sampled IDs and metadata, and check the distribution of `exp_year` (including how many CVs sit at the 20-year cap).

Logic: `src/download_data.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import utils

utils.set_seed()
print("SEED:", utils.SEED, "| device:", utils.get_device(), "| root:", utils.ROOT)

## How the sample is built

`src/download_data.py` does the work; this notebook only reads its outputs. Run it once from the repository root:

```bash
python -m src.download_data          # 800 CVs per occupation (default)
python -m src.download_data --n 5    # quick test
```

For each of the 5 occupations it streams the image shards from Hugging Face (`sumin-yu/PopResume`), keeps only **no-photo** images (`has_photo == False`), shuffles with **seed 42** (buffer of 5,000 CVs) and takes the first N. The run is resumable: finished occupations are skipped and existing images are not rewritten.

Outputs:
- `data/raw/images/<id>.png`: the CV images (not committed)
- `data/raw/metadata.csv`: `id`, `occupation`, `exp_year` and `sex` / `race`. `sex` and `race` are only for the fairness check (Question 4) and never used as features. `age`, names, `state` and `region` are not stored.
- `data/raw/sampled_ids.csv`: the sampled IDs, for reproducibility
- `data/raw/reference_text.csv`: `id` + reference `text`, used **only** to measure OCR quality

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.visualization import plot_exp_year_distribution

meta = pd.read_csv(utils.RAW_DIR / "metadata.csv", dtype={"id": str})
stats = json.loads((utils.METRICS_DIR / "download_stats.json").read_text())
print(f"{len(meta)} CVs sampled, {stats['n_per_occupation']} per occupation, seed {stats['seed']}")
print(f"Download time: {stats['total_seconds'] / 60:.1f} min · mean image size {stats['mean_image_kb']} KB · total {stats['total_images_mb']} MB")
meta["occupation"].value_counts().to_frame("CVs")

## Distribution of years of experience

`exp_year` is a continuous value (fractional years). The table summarises it overall and per occupation, before clipping. The last two columns count how many CVs sit at or above the 20-year cap: if more than 15% do, the assignment allows raising the cap.

In [ ]:
summary = pd.read_csv(utils.TABLES_DIR / "exp_year_distribution.csv")
summary

A histogram with 1-year bins is the most direct way to see the shape of the target: where most CVs sit, how long the right tail is, and how much mass the cap at 20 will absorb (dashed line). The small panels repeat it per occupation, because the split and the error analysis are stratified by occupation.

In [ ]:
fig = plot_exp_year_distribution(meta)
plt.show()

### Cap check

Values above 20 become 20 ("20 or more"), and predictions are clipped to the same range. The cell below reports the share at the cap and whether the 15% rule is triggered.

In [ ]:
n_cap = int((meta["exp_year"] >= utils.EXP_CAP).sum())
pct_cap = 100 * n_cap / len(meta)
print(f"{n_cap} of {len(meta)} CVs ({pct_cap:.1f}%) have exp_year >= {utils.EXP_CAP}")
print("Above 15%: consider raising the cap" if pct_cap > 15 else f"Below 15%: keep the cap at {utils.EXP_CAP}")

## Experience bands

The assignment scores band accuracy on the bands **0–2, 3–5, 6–10 and 11–20**. Those labels assume whole years, but `exp_year` is continuous (e.g. 2.6 years), so integer bands would leave gaps (where does 2.6 go?).

**Decision:** the bands are defined in `src/utils.py` as gap-free, left-closed intervals on the clipped target:

| Band | Interval | Meaning |
|---|---|---|
| 0–2 | [0, 3) | less than 3 years |
| 3–5 | [3, 6) | 3 to under 6 years |
| 6–10 | [6, 11) | 6 to under 11 years |
| 11–20 | [11, 20] | 11 years or more (20 = "20 or more" after clipping) |

So 2.99 years is band 0–2 and 3.0 is band 3–5, the same as truncating to whole completed years, which is how a recruiter counts experience. Every model's predictions are clipped to [0, 20] and mapped with the same function, `utils.to_band`. The table below shows how the sample falls into the bands, overall and per occupation.

In [ ]:
meta["band"] = pd.Categorical(utils.to_band(meta["exp_year"]), categories=utils.BAND_LABELS, ordered=True)
band_counts = pd.crosstab(meta["occupation"], meta["band"], margins=True, margins_name="all")
band_counts

## Example CVs

Two random sampled CVs, to see what PaddleOCR will have to read: layout, fonts and where the job dates are.

In [ ]:
examples = meta.sample(2, random_state=utils.SEED)
fig, axes = plt.subplots(1, 2, figsize=(12, 9))
for ax, (_, row) in zip(axes, examples.iterrows()):
    ax.imshow(Image.open(utils.RAW_IMAGES_DIR / f"{row['id']}.png"))
    ax.set_title(f"id {row['id']} · {row['occupation']} · {row['exp_year']:.1f} years", loc="left")
    ax.axis("off")
    ax.grid(False)
plt.tight_layout()
plt.show()

## Reference text coverage

Every sampled CV should have its reference text. It is never a model input: it is only used in Stage 2 to compute CER / WER and in Question 2.

In [ ]:
ref = pd.read_csv(utils.RAW_DIR / "reference_text.csv", dtype={"id": str})
print(f"Reference text for {ref['id'].isin(meta['id']).sum()} of {len(meta)} sampled CVs")
ref["text"].str.len().describe().round(0).to_frame("characters")